<a href="https://colab.research.google.com/github/Hasan-ari/chemical_thesis_repo/blob/main/conditional_model_v2_8_rocks/colab_eight_rocks_loro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sekiz Kayaç: Referans Eğitim + Leave-One-Rock-Out (LORO)

Bu defter iki iş yapar:

1. **Referans model:** 8 kayacın tamamı (55.122 koşu) tek LSTM'de, `rock_aware_run_level` bölme (%80/%10/%10, seed 42).
2. **LORO:** her kayaç için ayrı bir model; o kayaç eğitimden/validasyondan tamamen çıkar, sadece testte görünür (`test_unseen`). 8 kat = 8 model.

Kod `conditional_model_v1/` paketinden gelir (git'ten klonlanır). Bu klasördeki tek şey bu defter + README + hocanın ham dosyaları (git dışı).

Sözleşme (v2): `X = (koşu, 301, 58)` koşul + zaman, `Y = (koşu, 301, 26)` PHREEQC çıktı. Kayaç adı modele girmez.

Kayıt altına alınanlar: her koşu klasöründe `history.csv` (epoch başına train/val loss), `metrics.json`, `feature_metrics.csv`, `rock_feature_metrics.csv`, `plots/`; ortak `summary.csv` + `registry.sqlite` (`runs`, `rock_feature_metrics`, `loro_folds` tabloları); LORO için `loro_summary.csv`, `loro_rock_feature_metrics.csv`, `loro_metrics.json` ve toplu PNG'ler. Hepsi Drive'a kopyalanır.


## 0. Repoyu klonla

Her oturumda taze klon. Kod GitHub'da, veri Drive'da.

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

# Important: leave the repo folder before deleting it.
os.chdir("/content")

scheme = "https://"
host = "github" + ".com"
owner = "Hasan-ari"
repo = "chemical_thesis_repo"

REPO_URL = f"{scheme}{host}/{owner}/{repo}.git"
BRANCH = "main"
REPO_DIR = Path("/content/chemical_thesis_repo")

print("cwd before clone:", os.getcwd())
print("REPO_URL repr:", repr(REPO_URL))

shutil.rmtree(REPO_DIR, ignore_errors=True)

result = subprocess.run(
    ["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
    cwd="/content",
    capture_output=True,
    text=True,
)

print("returncode:", result.returncode)
print("STDOUT:")
print(result.stdout)
print("STDERR:")
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("git clone failed")

os.chdir(REPO_DIR)

print("cwd after clone:", os.getcwd())

print("HEAD:")
subprocess.run(["git", "rev-parse", "--short", "HEAD"], check=True)

print("Check data.py:")
subprocess.run(["ls", "-l", "conditional_model_v1/data.py"], check=True)

## 1. Ayarlar

Sadece burayı düzenle. Zip'ler **Shared with me** altındaysa MyDrive'a kısayol ekleyip yolları ona göre yaz.

In [ ]:
from __future__ import annotations

from pathlib import Path

REPO_DIR = Path("/content/chemical_thesis_repo")

# ---- Drive: ham veri zip'leri (git dışı) ----
DRIVE_DATA_DIR = Path("/content/drive/MyDrive/chemical_thesis_data")
ZIPS = {
    # zip dosyası -> (hedef üst klasör, config'in beklediği klasör adı)
    "Calcite_wat_sat_data_3.zip": ("wat_sat/Calcite", "Calcite_wat_sat_data_3"),
    "Dolomite_wat_sat_data_2.zip": ("wat_sat/Dolomite", "Dolomite_wat_sat_data_2"),
    "Halite_wat_sat_data_2.zip": ("wat_sat/Halite", "Halite_wat_sat_data_2"),
    "Trona_par_sat_data_3.zip": ("wat_sat/Trona", "Trona_par_sat_data_3"),
    "Sandstone_data_2.zip": ("new_data", "Sandstone_data_2"),
    "Quarzite_schist_low_t.zip": ("new_data", "Quarzite_schist_low_t"),
    "Mica_schist_low_t.zip": ("new_data", "Mica_schist_low_t"),
    "Mica_carbonate_schist_low_t.zip": ("new_data", "Mica_carbonate_schist_low_t"),
}

# ---- Colab yerel çalışma klasörleri (hızlı disk) ----
DATA_ROOT = Path("/content/data")
PROCESSED_ROOT = Path("/content/processed")
RUN_ROOT = Path("/content/runs")

# ---- Drive: kalıcı çıktılar ----
DRIVE_PROCESSED_ROOT = DRIVE_DATA_DIR / "processed"
DRIVE_RUN_ROOT = Path("/content/drive/MyDrive/chemical_thesis_repo/runs")

# ---- Config'ler (repo içinde; deney = config + commit) ----
REFERENCE_CONFIG = "configs/conditional_model_v1/full_colab_eight_rocks.yaml"
LORO_CONFIG = "configs/conditional_model_v1/loro_colab_eight_rocks.yaml"
CACHE_NAME = "eight_rocks_v1"  # iki config'in data.cache_name değeriyle aynı olmalı

# ---- LORO seçenekleri ----
# None -> config'deki held_out_rocks ("all" = 8 kat). Oturum yetmezse alt küme:
# HELD_OUT_ROCKS = ["Calcite", "Dolomite", "Halite", "Trona"]
HELD_OUT_ROCKS = None
# Referans koşu klasörü. None -> bu defterde eğitilen referans koşu kullanılır.
# Daha önce bitmiş bir koşu varsa Drive yolunu ver, örn:
# REFERENCE_RUN_DIR = DRIVE_RUN_ROOT / "20260905_101500_eight_rocks_condition_lstm_v1"
REFERENCE_RUN_DIR = None

print("REFERENCE_CONFIG:", REFERENCE_CONFIG)
print("LORO_CONFIG:", LORO_CONFIG)
print("CACHE_NAME:", CACHE_NAME)


## 2. Drive'ı bağla

In [ ]:
from google.colab import drive

drive.mount("/content/drive")


## 3. Bağımlılıklar

In [ ]:
import os
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)


## 4. Sekiz zip'i Colab diskine aç

Hiçbir veri atlanmaz: 8 zip'in hepsi zorunlu, biri eksikse hücre durur.

In [ ]:
import shutil
import tempfile
import zipfile
from pathlib import Path


def find_dataset_root(root: Path) -> Path:
    """input/ ve output/ klasörlerini gerçekten içeren klasörü bul."""
    candidates = [root] + [path for path in root.rglob("*") if path.is_dir()]
    for path in candidates:
        if (path / "input").is_dir() and (path / "output").is_dir():
            return path
    raise FileNotFoundError(f"input/ ve output/ içeren klasör bulunamadı: {root}")


def extract_dataset(zip_path: Path, target_parent: Path, expected_folder: str) -> Path:
    if not zip_path.exists():
        raise FileNotFoundError(f"Zip bulunamadı: {zip_path}")
    expected_path = target_parent / expected_folder
    shutil.rmtree(expected_path, ignore_errors=True)
    target_parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory() as tmp_dir:
        tmp_root = Path(tmp_dir)
        with zipfile.ZipFile(zip_path) as archive:
            archive.extractall(tmp_root)
        shutil.copytree(find_dataset_root(tmp_root), expected_path)
    n_in = len(list((expected_path / "input").glob("*_Input.txt")))
    n_out = len(list((expected_path / "output").glob("*_Output.txt")))
    print(f"Hazır: {expected_path}  input={n_in}  output={n_out}")
    return expected_path


missing = [name for name in ZIPS if not (DRIVE_DATA_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f"Drive'da eksik zip'ler: {missing} (klasör: {DRIVE_DATA_DIR})")

for zip_name, (sub_dir, expected_folder) in ZIPS.items():
    extract_dataset(DRIVE_DATA_DIR / zip_name, DATA_ROOT / sub_dir, expected_folder)


## 5. Ortam değişkenleri, kayıt defteri ve GPU

YAML'lar `${DATA_ROOT}` gibi değişkenler kullanır. Drive'daki `summary.csv` + `registry.sqlite` devam ettirilir (geçmiş silinmez).

In [ ]:
os.environ["DATA_ROOT"] = str(DATA_ROOT)
os.environ["PROCESSED_ROOT"] = str(PROCESSED_ROOT)
os.environ["RUN_ROOT"] = str(RUN_ROOT)

RUN_ROOT.mkdir(parents=True, exist_ok=True)
PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_RUN_ROOT.mkdir(parents=True, exist_ok=True)

for registry_name in ("summary.csv", "registry.sqlite"):
    drive_registry = DRIVE_RUN_ROOT / registry_name
    local_registry = RUN_ROOT / registry_name
    if drive_registry.exists() and not local_registry.exists():
        shutil.copy2(drive_registry, local_registry)
        print("Drive'dan alındı:", registry_name)

import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


## 6. Sekiz-kayaç önbelleğini kur veya Drive'dan al

TXT okuma tek seferlik yavaş adım. Drive'da doğrulanmış `eight_rocks_v1` varsa kopyalanır; yoksa kurulup Drive'a yazılır. Eğitim yok.

In [ ]:
LOCAL_CACHE = PROCESSED_ROOT / CACHE_NAME
DRIVE_CACHE = DRIVE_PROCESSED_ROOT / CACHE_NAME
SUCCESS_MARKER = "_SUCCESS"
cache_command = [sys.executable, "-m", "conditional_model_v1.cli.cache"]

if (DRIVE_CACHE / SUCCESS_MARKER).is_file():
    shutil.rmtree(LOCAL_CACHE, ignore_errors=True)
    LOCAL_CACHE.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(DRIVE_CACHE, LOCAL_CACHE)
    subprocess.run(cache_command + ["validate", "--config", REFERENCE_CONFIG, "--deep"], check=True)
    print(f"Drive önbelleği kullanıldı: {DRIVE_CACHE}")
else:
    if DRIVE_CACHE.exists():
        raise RuntimeError(f"Yarım kalmış Drive önbelleği var, elle sil: {DRIVE_CACHE}")
    subprocess.run(cache_command + ["prepare", "--config", REFERENCE_CONFIG], check=True)
    subprocess.run(cache_command + ["validate", "--config", REFERENCE_CONFIG, "--deep"], check=True)
    DRIVE_PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)
    shutil.copytree(LOCAL_CACHE, DRIVE_CACHE, ignore=shutil.ignore_patterns(SUCCESS_MARKER))
    shutil.copy2(LOCAL_CACHE / SUCCESS_MARKER, DRIVE_CACHE / SUCCESS_MARKER)
    local_processed_root = os.environ["PROCESSED_ROOT"]
    try:
        os.environ["PROCESSED_ROOT"] = str(DRIVE_PROCESSED_ROOT)
        subprocess.run(cache_command + ["validate", "--config", REFERENCE_CONFIG, "--deep"], check=True)
    finally:
        os.environ["PROCESSED_ROOT"] = local_processed_root
    print(f"Önbellek kuruldu ve Drive'a yazıldı: {DRIVE_CACHE}")


## 7. Referans model: 8 kayaç, tek LSTM

`REFERENCE_RUN_DIR` ayarlarda doluysa bu hücre atlanır (bitmiş koşu yeniden eğitilmez).

In [ ]:
def stream_command(command: list[str], marker: str) -> Path:
    """Alt süreci çalıştır, çıktıyı anlık yazdır, `marker=...` satırındaki yolu döndür."""
    process = subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    found = None
    for line in process.stdout:
        print(line, end="")
        if line.startswith(marker + "="):
            found = Path(line.split("=", 1)[1].strip())
    if process.wait() != 0:
        raise RuntimeError(f"Komut hata verdi: {' '.join(command)}")
    if found is None or not found.is_dir():
        raise RuntimeError(f"{marker}= satırı bulunamadı veya klasör yok")
    return found


def copy_run_to_drive(run_dir: Path) -> Path:
    """Koşu klasörü + summary.csv + registry.sqlite Drive'a. Var olan koşu ezilmez."""
    drive_run_dir = DRIVE_RUN_ROOT / run_dir.name
    if drive_run_dir.exists():
        raise FileExistsError(f"Drive'da aynı adlı koşu var, üzerine yazılmadı: {drive_run_dir}")
    shutil.copytree(run_dir, drive_run_dir)
    for registry_name in ("summary.csv", "registry.sqlite"):
        if (RUN_ROOT / registry_name).exists():
            shutil.copy2(RUN_ROOT / registry_name, DRIVE_RUN_ROOT / registry_name)
    print(f"Drive'a kopyalandı: {drive_run_dir}")
    return drive_run_dir


if REFERENCE_RUN_DIR is None:
    REFERENCE_RUN_DIR = stream_command(
        [sys.executable, "-m", "conditional_model_v1.cli.train", "--config", REFERENCE_CONFIG],
        "run_dir",
    )
    copy_run_to_drive(REFERENCE_RUN_DIR)
else:
    REFERENCE_RUN_DIR = Path(REFERENCE_RUN_DIR)
    if not (REFERENCE_RUN_DIR / "metrics.json").is_file():
        raise FileNotFoundError(f"Referans koşuda metrics.json yok: {REFERENCE_RUN_DIR}")
    print("Hazır referans koşu kullanılıyor:", REFERENCE_RUN_DIR)


## 8. Referans koşunun metrikleri ve 24 kayaç görseli

Her kayaç için best/worst/mean: 8 × 3 = 24 PNG. Loss eğrisi `plots/loss_curve.png`.

In [ ]:
import json

import pandas as pd
from IPython.display import Image, display

metrics = json.loads((REFERENCE_RUN_DIR / "metrics.json").read_text())
print("best_val_loss:", metrics["best_val_loss"], " rmse_mean_original:", metrics["rmse_mean_original"])
per_rock = pd.DataFrame.from_dict(metrics["per_rock"], orient="index")
display(per_rock[["n_runs", "rmse_mean_original", "rmse_mean_normalized", "best_run_id", "worst_run_id"]])
display(pd.read_csv(REFERENCE_RUN_DIR / "history.csv").tail())
display(Image(filename=str(REFERENCE_RUN_DIR / "plots" / "loss_curve.png")))

# Kutu grafikleri (box plot): her test koşusunun kendi RMSE'si -> dağılım
# Kaynak sayılar: run_rmse.csv ve registry.sqlite `run_rmse` tablosu
for name in ("rmse_boxplot_by_rock.png", "rmse_boxplot_by_feature.png",
             "rmse_boxplot_by_rock_and_feature.png"):
    print(name)
    display(Image(filename=str(REFERENCE_RUN_DIR / "plots" / "boxplots" / name)))

overview_paths = sorted((REFERENCE_RUN_DIR / "plots" / "rock_overviews").glob("*_overview.png"))
expected = 3 * len(per_rock)
if len(overview_paths) != expected:
    raise RuntimeError(f"{expected} overview PNG bekleniyordu, {len(overview_paths)} bulundu")
for overview_path in overview_paths:
    print(overview_path.name)
    display(Image(filename=str(overview_path)))


## 9. LORO: 8 kat, 8 model

Her kat bittiğinde `run_dir=` satırı gelir ve o kat hemen Drive'a kopyalanır; Colab kopsa bile biten katlar kaybolmaz. `loro_summary.csv` her kattan sonra yeniden yazılır.

Süre: her kat ≈ referans koşunun 7/8'i. Oturum yetmezse ayarlarda `HELD_OUT_ROCKS` ile alt küme ver, sonra kalanlarla tekrar çalıştır (sqlite `loro_folds` tablosu birikir).

In [ ]:
loro_command = [sys.executable, "-m", "conditional_model_v1.cli.loro", "--config", LORO_CONFIG,
                "--reference-run-dir", str(REFERENCE_RUN_DIR)]
if HELD_OUT_ROCKS:
    loro_command += ["--held-out-rocks", *HELD_OUT_ROCKS]
print("Komut:", " ".join(loro_command))

process = subprocess.Popen(
    loro_command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)
LORO_DIR = None
FOLD_RUN_DIRS = []
for line in process.stdout:
    print(line, end="")
    if line.startswith("run_dir="):
        fold_dir = Path(line.split("=", 1)[1].strip())
        FOLD_RUN_DIRS.append(fold_dir)
        copy_run_to_drive(fold_dir)  # kat biter bitmez Drive'a
    elif line.startswith("loro_dir="):
        LORO_DIR = Path(line.split("=", 1)[1].strip())
if process.wait() != 0:
    raise RuntimeError("LORO hata ile bitti; yukarıdaki çıktıya bak. Biten katlar Drive'da.")
if LORO_DIR is None or not LORO_DIR.is_dir():
    raise RuntimeError("loro_dir= satırı bulunamadı")

DRIVE_LORO_DIR = copy_run_to_drive(LORO_DIR)
print(f"{len(FOLD_RUN_DIRS)} kat + toplu klasör Drive'da: {DRIVE_LORO_DIR}")


## 10. LORO sonuçları

`unseen` = modelin hiç görmediği kayaçtaki hata; `seen` = aynı modelin diğer kayaçların test payındaki hatası; `reference` = 8-kayaç referans modelin o kayaçtaki hatası. Orijinal birim sütunları katlar arası doğrudan karşılaştırılabilir; normalize sütunlar her katın kendi ölçekleyicisiyle hesaplanır.

In [ ]:
import sqlite3

summary = pd.read_csv(LORO_DIR / "loro_summary.csv")
display(summary[[
    "held_out_rock", "n_train", "n_val", "n_test_unseen", "n_test_seen",
    "unseen_rmse_mean_original", "seen_rmse_mean_original", "reference_rmse_mean_original",
    "unseen_rmse_mean_normalized", "seen_rmse_mean_normalized", "reference_rmse_mean_normalized",
    "unseen_over_reference_normalized", "best_val_loss",
]])

with sqlite3.connect(RUN_ROOT / "registry.sqlite") as connection:
    display(pd.read_sql_query(
        "SELECT loro_name, held_out_rock, n_test_unseen, unseen_rmse_mean_original, "
        "seen_rmse_mean_original, best_val_loss FROM loro_folds ORDER BY loro_name, held_out_rock",
        connection,
    ))

# loro_unseen_rmse_boxplot.png: dışarıda bırakılan kayaçta koşu başına hata dağılımı
# (mavi = o kayacı hiç görmemiş LORO modeli, turuncu = 8 kayaçlı referans model)
for name in ("loro_rmse_original_by_rock.png", "loro_rmse_normalized_by_rock.png",
             "loro_unseen_rmse_normalized_heatmap.png", "loro_unseen_rmse_boxplot.png"):
    print(name)
    display(Image(filename=str(LORO_DIR / "plots" / name)))

# Her katın loss eğrisi + dışarıda bırakılan kayacın en kötü örneği
for fold_dir in FOLD_RUN_DIRS:
    print(fold_dir.name)
    display(Image(filename=str(fold_dir / "plots" / "loss_curve.png")))
    display(Image(filename=str(fold_dir / "plots" / "boxplots" / "rmse_boxplot_by_rock.png")))
for overview_path in sorted((LORO_DIR / "plots" / "rock_overviews").glob("*_worst_overview.png")):
    print(overview_path.name)
    display(Image(filename=str(overview_path)))
